In [1]:
using LinearAlgebra
using Ket
using JLD2
using ProgressMeter
using Random
using MosekTools

using ppt2


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
# ── Projector from a product vector ──────────────────────────────────────────

"""Rank-1 projector onto the product state |a⟩⊗|b⟩."""
function product_projector(a::Vector, b::Vector)
    v = kron(a, b)
    return v * v'
end

product_projector

In [3]:
# ── UPB → Bound Entangled State ───────────────────────────────────────────────

"""
Given an UPB as a vector of (a, b) pairs (normalised kets),
return the bound entangled state ρ = (I - Π) / (d - |UPB|)
where Π = sum of rank-1 projectors onto the UPB vectors.
"""
function upb_state(upb::Vector{Tuple{Vector{Float64},Vector{Float64}}}, dA::Int, dB::Int)
    d  = dA * dB
    Π  = sum(product_projector(a, b) for (a, b) in upb)
    ρ  = (I(d) - Π) / (d - length(upb))
    return Hermitian(ρ)
end

upb_state

In [4]:
# ── Random UPB search (small dimensions) ──────────────────────────────────────
#
#   Numerically search for UPBs by growing a set of mutually orthogonal
#   product states until no orthogonal product complement exists.
#   Uses randomised restarts.

"""
Check if a candidate product vector |a⟩⊗|b⟩ is orthogonal to all states
in `upb` (list of (a,b) pairs).
"""
function orthogonal_to_all(a, b, upb)
    v = kron(a, b)
    return all(abs(dot(kron(u_a, u_b), v)) < 1e-8 for (u_a, u_b) in upb)
end

"""
Attempt to find a product vector orthogonal to the given UPB partial set
by random sampling + projection onto the orthogonal complement.
Returns (found::Bool, a, b).
"""
function find_orthogonal_product_vector(upb, dA, dB; n_trials=5000)
    d = dA * dB
    # Build orthogonal complement projector
    if isempty(upb)
        return true, normalize(randn(dA)), normalize(randn(dB))
    end
    V = hcat([normalize(kron(a, b)) for (a,b) in upb]...)
    P_perp = I(d) - V * V'   # projector onto ⊥ of span(UPB)

    for _ in 1:n_trials
        a0 = normalize(randn(dA))
        b0 = normalize(randn(dB))
        # Project onto complement and try to separate
        v  = normalize(P_perp * (kron(a0, b0)))
        # Schmidt rank-1 check via SVD of reshaped vector
        M  = reshape(v, dA, dB)
        sv = svdvals(M)
        if sv[1] / sum(sv) > 1 - 1e-6   # nearly rank-1
            a_new = normalize(M * M' * a0)  # dominant left singular vector
            b_new = normalize(M' * a_new)
            if orthogonal_to_all(a_new, b_new, upb)
                return true, a_new, b_new
            end
        end
    end
    return false, zeros(dA), zeros(dB)
end

"""
Numerically construct a random UPB in C^dA ⊗ C^dB.
Grows the set greedily; returns the UPB and the resulting bound entangled state.
"""
function random_upb(dA::Int, dB::Int; max_size=nothing, n_trials=5000, verbose=false)
    upb = Tuple{Vector{Float64}, Vector{Float64}}[]
    max_size = something(max_size, dA * dB - 1)

    for step in 1:max_size
        found, a, b = find_orthogonal_product_vector(upb, dA, dB; n_trials)
        if !found
            verbose && println("UPB complete at size $(length(upb)) (step $step)")
            break
        end
        push!(upb, (normalize(a), normalize(b)))
        verbose && println("Step $step: added product vector, UPB size = $(length(upb))")
    end

    ρ = upb_state(upb, dA, dB)
    return upb, ρ
end

random_upb

In [5]:
# ── Verification ──────────────────────────────────────────────────────────────

"""Check if a matrix is PPT (all eigenvalues of partial transpose ≥ -tol)."""
function is_ppt(ρ::Matrix, dA::Int, dB::Int; tol=1e-8)
    PT = partial_transpose(ρ, 2)
    return all(eigvals(Hermitian(PT)) .≥ -tol)
end

"""
Realignment criterion: if ‖R(ρ)‖₁ > 1, state is entangled.
R(ρ)[ia, jb] = ρ[ij, ab]  (reshape indices).
"""
function realignment_norm(ρ::Matrix, dA::Int, dB::Int)
    n = dA * dB
    R = zeros(ComplexF64, n, n)
    for i in 0:dA-1, a in 0:dB-1, j in 0:dA-1, b in 0:dB-1
        R[i*dA+j+1, a*dB+b+1] = ρ[i*dB+a+1, j*dB+b+1]
    end
    return sum(svdvals(R))  # nuclear norm = sum of singular values
end

"""Print a summary of entanglement properties of ρ in C^dA ⊗ C^dB."""
function verify_bound_entangled(ρ::AbstractMatrix, dA::Int, dB::Int; name="ρ")
    println("── $name ($( dA)⊗$dB, dim $(size(ρ,1))×$(size(ρ,2))) ──")
    
    eigs_ρ  = eigvals(Hermitian(Matrix(ρ)))
    PT      = partial_transpose(Matrix(ρ), 2)
    eigs_PT = eigvals(Hermitian(PT))
    ra_norm = realignment_norm(Matrix(ρ), dA, dB)
    tr_ρ    = tr(ρ)
    purity  = real(tr(ρ^2))

    println("  Trace:               $(round(real(tr_ρ), digits=6))")
    println("  Purity tr(ρ²):       $(round(purity, digits=6))")
    println("  Min eig(ρ):          $(round(minimum(eigs_ρ), digits=8))  $(minimum(eigs_ρ) ≥ -1e-8 ? "✓ PSD" : "✗ not PSD")")
    println("  Min eig(ρᴳ):         $(round(minimum(eigs_PT), digits=8)) $(minimum(eigs_PT) ≥ -1e-8 ? "✓ PPT" : "✗ not PPT")")
    println("  Realignment ‖R(ρ)‖₁: $(round(ra_norm, digits=6))          $(ra_norm > 1 + 1e-6 ? "→ entangled (realignment)" : "→ inconclusive")")
    println("  Verdict:             PPT=$(is_ppt(Matrix(ρ),dA,dB)) — bound entangled by UPB range criterion")
    println()
end

verify_bound_entangled

In [6]:
upb, rho = random_upb(4, 4)
rho

16×16 Hermitian{Float64, Matrix{Float64}}:
  0.0666656     8.65614e-8   5.03418e-6   …   0.000167915  -0.000150011
  8.65614e-8    0.0666667   -4.18704e-7      -1.39658e-5    1.24768e-5
  5.03418e-6   -4.18704e-7   0.0666423       -0.000812216   0.000725616
 -4.49743e-6    3.74061e-7   2.17544e-5       0.000725616  -0.000648249
  1.3818e-5    -1.14927e-6  -6.68387e-5      -0.0022294     0.0019917
 -1.14927e-6    9.55875e-8   5.55912e-6   …   0.000185424  -0.000165654
 -6.68387e-5    5.55912e-6   0.000323303      0.0107838    -0.00963398
  5.97122e-5   -4.96639e-6  -0.000288832     -0.00963398    0.00860679
  1.46133e-5   -1.21542e-6  -7.06855e-5      -0.00235771    0.00210633
 -1.21542e-6    1.01089e-7   5.87907e-6       0.000196096  -0.000175188
 -7.06855e-5    5.87907e-6   0.000341911  …   0.0114044    -0.0101885
  6.31489e-5   -5.25223e-6  -0.000305456     -0.0101885     0.00910214
 -3.47142e-5    2.88725e-6   0.000167915      0.00560079   -0.00500362
  2.88725e-6   -2.40139e-7  -1.

In [7]:
verify_bound_entangled(rho, 4, 4)

── ρ (4⊗4, dim 16×16) ──
  Trace:               1.0
  Purity tr(ρ²):       0.066667
  Min eig(ρ):          0.0  ✓ PSD
  Min eig(ρᴳ):         -0.0 ✓ PPT
  Realignment ‖R(ρ)‖₁: 0.305505          → inconclusive
  Verdict:             PPT=true — bound entangled by UPB range criterion



In [8]:
forms = jldopen("../pncp_forms_4x4.jld2") do file
    vcat([file[k] for k in keys(file)]...)
end
forms ./= tr.(forms);

In [9]:
forms[1500]

16×16 Matrix{Float64}:
  0.156289     -0.168171     -0.038248    …  -0.0263394    0.025308
 -0.168171      0.184072      0.0466496       0.0371019   -0.0227138
 -0.038248      0.0466496     0.0742756       0.022067    -0.0159187
 -0.00154951   -0.00542052   -0.0459364      -0.0159187    0.000441867
 -0.0473574     0.000123715   0.0380506       0.0146423   -0.00709966
  0.000123715   0.0558709    -0.0134824   …  -0.00871482  -0.0212992
  0.0380506    -0.0134824    -0.0417733      -0.0250607    0.00683867
  0.0252523    -0.0345756    -0.00182437      0.00683867   0.0198068
 -0.0451552     0.0943049     0.00937021      0.00512951  -0.00386504
  0.0943049    -0.146331     -0.0342931      -0.00940354   0.0343165
  0.00937021   -0.0342931    -0.0124616   …  -0.0137926    0.00817575
 -0.0302691     0.034218      0.023531        0.00817575  -0.0242936
  0.0356687    -0.0555276    -0.0263394      -0.0101066    0.00402359
 -0.0555276     0.0711269     0.0371019       0.00819647  -0.023542
 -0.02

In [26]:
_, ppt_1 = random_upb(4, 4)
_, ppt_2 = random_upb(4, 4)
composite = Hermitian(ampliation(ppt_1, ppt_2, 4, 4))
println("eigmin rho: ", eigmin(composite))
println("eigmin tr.: ", eigmin(partial_transpose(composite, 1)))
r, w = entanglement_robustness(composite, [4, 4], 2; solver=Mosek.Optimizer)
println("Robustness: ", r)
println("min trace : ", findmin(tr.(forms .* Ref(composite))))
println("Ampliation: ", findmin(minimum.(real.(eigvals.(ampliation.(forms, Ref(composite), 4, 4))))))

eigmin rho: 0.008913532175462374
eigmin tr.: 0.008913532175462374
Robustness: -0.008913532176632029
min trace : (0.01375805302845923, 1467)
Ampliation: (1.3307240658164125e-5, 1500)


In [33]:
_, ppt = random_upb(4, 4)
println("eigmin rho: ", eigmin(ppt))
println("eigmin tr.: ", eigmin(partial_transpose(ppt, 1)))
r, w = entanglement_robustness(ppt, [4, 4], 2; solver=Mosek.Optimizer)
println("Robustness: ", r)
println("min trace : ", findmin(tr.(forms .* Ref(ppt))))
println("Ampliation: ", findmin(minimum.(real.(eigvals.(ampliation.(forms, Ref(ppt), 4, 4))))))

eigmin rho: -1.4611559252608962e-17
eigmin tr.: -1.1523455174302643e-18
Robustness: 2.409298608208429e-13
min trace : (0.04278284995917862, 579)
Ampliation: (5.507919391755357e-5, 1500)


In [34]:
composite = Hermitian(ampliation(ppt, ppt, 4, 4))
println("eigmin rho: ", eigmin(composite))
println("eigmin tr.: ", eigmin(partial_transpose(composite, 1)))
r, w = entanglement_robustness(composite, [4, 4], 2; solver=Mosek.Optimizer)
println("Robustness: ", r)
println("min trace : ", findmin(tr.(forms .* Ref(composite))))
println("Ampliation: ", findmin(minimum.(real.(eigvals.(ampliation.(forms, Ref(composite), 4, 4))))))

eigmin rho: 0.008905315139476115
eigmin tr.: 0.008905315139476115
Robustness: -0.00890531513955936
min trace : (0.012891728934479202, 579)
Ampliation: (1.3601437205747074e-5, 1500)


In [313]:
function test_ppt2_upb(n::Int, m::Int, forms; n_trials::Int=10000, tol::Float64=1e-8)
    @showprogress for i in 1:n_trials
        upb, ppt = random_upb(n, m)
        composite = Hermitian(ampliation(ppt, ppt, 4, 4))
        r1, w = entanglement_robustness(composite, [4, 4], 2; solver=Mosek.Optimizer)
        r2, idx2 = findmin(tr.(forms .* Ref(composite)))
        r3, idx3 = findmin(minimum.(real.(eigvals.(ampliation.(forms, Ref(composite), 4, 4)))))
        if r1 > tol || r2 < -tol || r3 < -tol
            println("DPS detects entanglement")
            println("eigmin rho: ", eigmin(composite))
            println("eigmin tr.: ", eigmin(partial_transpose(composite, 1)))
            println("Robustness: ", r1)
            println("min trace : ", r2)
            println("Ampliation: ", r3)
            return ppt, [w, forms[idx2], forms[idx3]]
        end
    end
    return nothing, nothing
end

test_ppt2_upb (generic function with 1 method)

In [314]:
ppt, form = test_ppt2_upb(4, 4, forms)

Progress:  45%|██████████████████▎                      |  ETA: 1:46:21Excessive output truncated after 524376 bytes.

(nothing, nothing)